# 🏥 MedGemma-Seg: Segmentación Multimodal de Imágenes Médicas

Este notebook ejecuta el pipeline **MedGemma-Seg** que combina:
1. **MedGemma 4B-IT** — Genera texto diagnóstico a partir de la imagen médica
2. **SAM 2 Tiny** — Genera máscaras candidatas (segmentación)
3. **MedSigLIP** — Compara la descripción con cada máscara para seleccionar la mejor

### Requisitos Previos
1. Aceptar los términos de uso de [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it)
2. Aceptar los términos de uso de [google/medsiglip-448](https://huggingface.co/google/medsiglip-448)
3. Tener un [Access Token de HuggingFace](https://huggingface.co/settings/tokens)
4. Usar un runtime con **GPU T4** mínimo (Runtime → Change runtime type → T4 GPU)

### VRAM Estimado: ~10 GB (cabe en T4 16 GB)
| Modelo | VRAM |
|--------|------|
| MedGemma 4B-IT (bf16) | ~8 GB |
| SAM 2 Tiny | ~0.3 GB |
| MedSigLIP (vision) | ~1.5 GB |
| SigLIP (text only) | ~0.4 GB |

---
## Paso 0: Verificar GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU detectada: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if vram_gb < 14:
        print("⚠️ Advertencia: Se recomienda >= 16 GB VRAM (T4 o superior)")
else:
    print("❌ No hay GPU! Ve a Runtime > Change runtime type > GPU")
    raise RuntimeError("GPU requerida")

---
## Paso 1: Instalar Dependencias

In [ ]:
%%capture
!pip install torch torchvision transformers>=4.40.0 accelerate>=0.28.0 \
    sentencepiece protobuf einops huggingface_hub \
    Pillow numpy matplotlib scipy tqdm

---
## Paso 2: Instalar SAM 2 + Descargar Checkpoint

In [ ]:
import os
if not os.path.exists('/content/sam2'):
    !git clone https://github.com/facebookresearch/sam2.git /content/sam2
    %cd /content/sam2
    !pip install -e . -q
    %cd /content
else:
    print("SAM 2 ya instalado")

# Descargar checkpoint
CKPT_PATH = "/content/sam2/checkpoints/sam2.1_hiera_tiny.pt"
if not os.path.exists(CKPT_PATH):
    os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt \
        -O {CKPT_PATH}
    print(f"✅ Checkpoint descargado en {CKPT_PATH}")
else:
    print(f"✅ Checkpoint ya existe en {CKPT_PATH}")

---
## Paso 3: Subir el Proyecto

**Opción A:** Clonar desde GitHub (descomentar y ajustar la URL)  
**Opción B:** Subir un .zip del proyecto manualmente

In [ ]:
# === OPCIÓN A: Clonar desde GitHub ===
# Descomenta y pon la URL de tu repositorio:
# !git clone https://github.com/TU_USUARIO/Medgemma_Segmentation.git /content/project

# === OPCIÓN B: Subir ZIP manualmente ===
# 1. Comprime la carpeta del proyecto como .zip
# 2. Ejecuta esta celda y sube el archivo
import os
if not os.path.exists('/content/project/opcion_A'):
    from google.colab import files
    print("Sube el archivo .zip del proyecto:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !mkdir -p /content/project
    !unzip -q {zip_name} -d /content/project
    # Si el zip crea un subdirectorio extra, ajustar:
    if not os.path.exists('/content/project/opcion_A'):
        subdirs = [d for d in os.listdir('/content/project') if os.path.isdir(f'/content/project/{d}')]
        if subdirs:
            !mv /content/project/{subdirs[0]}/* /content/project/
    print("✅ Proyecto cargado")
else:
    print("✅ Proyecto ya existe en /content/project")

# Agregar al path de Python
import sys
if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')
print(f"Archivos del proyecto: {os.listdir('/content/project/opcion_A/')}")

---
## Paso 4: Autenticación en HuggingFace

MedGemma y MedSigLIP son modelos con acceso restringido (gated).  
Necesitas un token de HuggingFace con permisos de lectura.

**Opción recomendada:** Usa los Secrets de Colab:  
1. Click en el 🔑 (llave) en el panel izquierdo
2. Agrega un secret llamado `HF_TOKEN` con tu token
3. Activa "Notebook access"

In [ ]:
from huggingface_hub import login

# Intenta usar Colab Secrets primero
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Autenticado via Colab Secrets")
except Exception:
    # Fallback: login interactivo
    print("No se encontró HF_TOKEN en Secrets. Login interactivo:")
    login()

---
## Paso 5: Cargar Imagen Médica

Sube una imagen médica (ej: fondo de ojo, radiografía, etc.).  
Formatos soportados: JPG, PNG, BMP.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Subir imagen
from google.colab import files
print("Sube una imagen médica:")
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]

# Cargar y mostrar
image = np.array(Image.open(IMAGE_PATH).convert("RGB"))
print(f"Imagen cargada: {IMAGE_PATH} — shape: {image.shape}")

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(f"Imagen: {IMAGE_PATH}")
plt.axis("off")
plt.show()

---
## Paso 6: Crear y Cargar el Pipeline

Esto descarga los modelos de HuggingFace (~5 min la primera vez).  
Los modelos se cachean en `/root/.cache/huggingface/`.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

from opcion_A.pipeline import MedGemmaSegPipeline

# Crear pipeline con ruta correcta al checkpoint de SAM 2
pipeline = MedGemmaSegPipeline(device="cuda")
pipeline.sam.checkpoint = "/content/sam2/checkpoints/sam2.1_hiera_tiny.pt"

# Cargar todos los modelos
pipeline.load()
print("\n✅ Todos los modelos cargados exitosamente")

# Verificar VRAM
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram_used:.1f} / {vram_total:.1f} GB")

---
## Paso 7: Ejecutar Inferencia (Modo Coseno)

Este es el modo baseline que no requiere support set.  
El pipeline:
1. Genera un análisis médico en texto (MedGemma)
2. Genera máscaras candidatas (SAM 2)
3. Selecciona la máscara más relevante al diagnóstico (MedSigLIP + coseno)

In [ ]:
# Puedes personalizar el prompt según la imagen
PROMPT = "Describe y segmenta la patología visible en esta imagen médica"

# Ejecutar pipeline
result = pipeline(image, prompt=PROMPT)

# Mostrar resultados
print("=" * 60)
print("RESULTADOS MedGemma-Seg")
print("=" * 60)
print(f"\n📝 Texto diagnóstico:\n{result['text']}")
print(f"\n🎭 Máscaras candidatas: {len(result['masks'])}")
print(f"⭐ Mejor score coseno: {result['best_score']:.4f}")
print(f"🏆 Índice mejor máscara: {result['best_idx']}")
print(f"\nTop 5 scores: {[f'{s:.3f}' for s in result['cosine_scores'][:5]]}")

---
## Paso 8: Visualizar Resultado

Muestra la imagen original, la máscara seleccionada, y el overlay con contorno.

In [ ]:
from opcion_A.visualization import create_overlay

# Crear visualización (3 paneles: original, máscara, overlay)
overlay = create_overlay(
    image,
    result['selected_mask'],
    result,
    output_path="resultado_medgemma_seg.png"
)

# Mostrar en notebook
from IPython.display import Image as IPImage, display
display(IPImage(filename="resultado_medgemma_seg.png"))

print(f"\n✅ Imagen guardada como 'resultado_medgemma_seg.png'")

---
## Paso 9 (Opcional): Visualizar Múltiples Máscaras Candidatas

Muestra las top-N máscaras candidatas con sus scores.

In [ ]:
import matplotlib.pyplot as plt

# Mostrar top 6 máscaras
top_n = min(6, len(result['masks']))
ranking = result['ranking'][:top_n]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, idx in enumerate(ranking):
    mask = result['masks'][idx]
    seg = mask['segmentation']
    score = result['cosine_scores'][idx]

    # Overlay rápido
    vis = image.copy().astype(float)
    vis[seg] = vis[seg] * 0.5 + np.array([0, 255, 128]) * 0.5

    axes[i].imshow(vis.clip(0, 255).astype(np.uint8))
    color = 'lime' if i == 0 else 'white'
    label = ' ⭐ SELECCIONADA' if i == 0 else ''
    axes[i].set_title(f"Rank #{i+1} (score: {score:.3f}){label}",
                      fontsize=12, color=color)
    axes[i].axis("off")

# Ocultar ejes extra si hay menos de 6
for j in range(top_n, len(axes)):
    axes[j].axis("off")

plt.suptitle("Top Máscaras Candidatas por Score Coseno", fontsize=14)
plt.tight_layout()
plt.show()

---
## Paso 10 (Opcional): Modo FSL/FD (Avanzado)

El modo FSL/FD requiere un **support set**: un conjunto de 6-12 embeddings
de máscaras de ejemplo de la patología a detectar.

Si no tienes un support set pre-computado, el pipeline usará modo coseno automáticamente.

In [ ]:
# Modo FSL/FD (requiere support set .pt)
# Si tienes un archivo .pt con embeddings de support set:

# support_embs = torch.load("support_set_cataract.pt", map_location="cuda")
# result_fsl = pipeline.run_with_fsl(image, support_embs, prompt=PROMPT)

# Sin support set, cae a modo coseno con advertencia:
result_fsl = pipeline.run_with_fsl(image, support_embeddings=None, prompt=PROMPT)

print(f"Modo: {'FSL/FD' if 'warning' not in result_fsl else 'Coseno (fallback)'}")
print(f"Score: {result_fsl.get('best_score', result_fsl.get('score', 'N/A'))}")
if 'warning' in result_fsl:
    print(f"⚠️ {result_fsl['warning']}")

---
## Paso 11: Liberar VRAM

In [ ]:
pipeline.unload()
import gc
gc.collect()
torch.cuda.empty_cache()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"✅ VRAM liberada. Uso actual: {vram_used:.2f} GB")

---
## Descargar Resultados

In [ ]:
from google.colab import files
files.download('resultado_medgemma_seg.png')